# W2/D2 RCA - Graph + Retrieval

Notebook này implement pipeline RCA không dùng LLM: graph traversal, temporal scoring, retrieval lịch sử incident và classifier kNN-style.


In [1]:
from pathlib import Path
import json
from datetime import datetime

cwd = Path.cwd()
if (cwd / "dataset").exists() and (cwd.parent / "d1" / "results" / "cluster_summary.json").exists():
    BASE = cwd
else:
    BASE = Path("w2/d2")

DATASET = BASE / "dataset"
D1_CLUSTER_SUMMARY = BASE.parent / "d1" / "results" / "cluster_summary.json"
RESULTS_DIR = BASE / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("Dataset files:", sorted(p.name for p in DATASET.iterdir()))


BASE: w2\d2
Dataset files: ['alerts_sample.jsonl', 'incidents_history.json', 'services.json']


In [2]:
def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

cluster_summary = load_json(D1_CLUSTER_SUMMARY)
services_data = load_json(DATASET / "services.json")
incidents_history = load_json(DATASET / "incidents_history.json")
alerts = load_jsonl(DATASET / "alerts_sample.jsonl")
alert_by_id = {alert["id"]: alert for alert in alerts}

print("clusters:", cluster_summary["output_clusters"])
print("alerts:", len(alerts))
print("history incidents:", len(incidents_history["incidents"]))


clusters: 3
alerts: 20
history incidents: 29


In [3]:
def parse_ts(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))

def build_graph(services_doc):
    nodes = {item["name"] for item in services_doc["services"]} | {item["name"] for item in services_doc["stores"]}
    graph = {node: set() for node in nodes}
    for edge in services_doc["edges"]:
        graph.setdefault(edge["from"], set()).add(edge["to"])
        graph.setdefault(edge["to"], set())
    return graph

graph = build_graph(services_data)
edge_count = sum(len(targets) for targets in graph.values())
print("graph nodes:", len(graph))
print("graph edges:", edge_count)


graph nodes: 14
graph edges: 17


In [4]:
def pagerank(nodes, edges, damping=0.85, iterations=60):
    nodes = sorted(nodes)
    n = len(nodes)
    if n == 0:
        return {}
    score = {node: 1.0 / n for node in nodes}
    outgoing = {node: [dst for src, dst in edges if src == node and dst in nodes] for node in nodes}
    incoming = {node: [src for src, dst in edges if dst == node and src in nodes] for node in nodes}

    for _ in range(iterations):
        dangling = sum(score[node] for node in nodes if not outgoing[node])
        next_score = {node: (1 - damping) / n + damping * dangling / n for node in nodes}
        for node in nodes:
            for src in incoming[node]:
                next_score[node] += damping * score[src] / len(outgoing[src])
        score = next_score
    return score

def first_alert_times(cluster, nodes):
    service_ts = {}
    for alert_id in cluster["alert_ids"]:
        alert = alert_by_id.get(alert_id)
        if alert and alert["service"] in nodes:
            ts = parse_ts(alert["ts"])
            service_ts[alert["service"]] = min(ts, service_ts.get(alert["service"], ts))
    return service_ts

def temporal_scores(cluster, nodes):
    service_ts = first_alert_times(cluster, nodes)
    if len(nodes) == 1:
        return {next(iter(nodes)): 0.55}, service_ts
    if not service_ts:
        return {node: 0.5 for node in nodes}, service_ts
    min_ts = min(service_ts.values())
    max_ts = max(service_ts.values())
    span = (max_ts - min_ts).total_seconds()
    scores = {}
    for node in nodes:
        if node not in service_ts:
            scores[node] = 0.0
        elif span <= 0:
            scores[node] = 1.0
        else:
            scores[node] = 1.0 - ((service_ts[node] - min_ts).total_seconds() / span)
    return scores, service_ts

def graph_candidates(cluster):
    nodes = set(cluster["services"])
    subgraph = [(src, dst) for src, targets in graph.items() for dst in targets if src in nodes and dst in nodes]
    pr = pagerank(nodes, subgraph)
    max_pr = max(pr.values()) if pr else 1.0
    graph_norm = {node: (pr.get(node, 0.0) / max_pr if max_pr else 0.0) for node in nodes}
    ts_score, service_ts = temporal_scores(cluster, nodes)

    candidates = []
    for node in sorted(nodes):
        final_score = 0.55 if len(nodes) == 1 else 0.6 * graph_norm.get(node, 0.0) + 0.4 * ts_score.get(node, 0.0)
        candidates.append({
            "service": node,
            "score": round(final_score, 4),
            "graph_score": round(graph_norm.get(node, 0.0), 4),
            "timestamp_score": round(ts_score.get(node, 0.0), 4),
            "first_alert": service_ts.get(node).isoformat().replace("+00:00", "Z") if node in service_ts else None,
        })
    return sorted(candidates, key=lambda item: (item["score"], item["graph_score"], item["timestamp_score"]), reverse=True)[:3]

for cluster in cluster_summary["clusters"]:
    print(cluster["cluster_id"], [(item["service"], item["score"]) for item in graph_candidates(cluster)])


c-001-000 [('payment-svc', 0.8943), ('checkout-svc', 0.8385), ('cart-svc', 0.5604)]
c-001-001 [('recommender-svc', 0.55)]
c-001-002 [('search-svc', 0.55)]


In [5]:
def normalize_severity(value):
    return {"crit": "critical", "warn": "medium"}.get(value, value)

def cluster_has_pool_signal(cluster):
    text = " ".join(cluster.get("fingerprints", [])).lower()
    return "pool" in text or "db_connection" in text

def retrieval_score(cluster, incident):
    cluster_services = set(cluster["services"])
    score = 0.0
    if incident["root_cause_service"] in cluster_services:
        score += 0.4
    overlap = len(cluster_services & set(incident["services_involved"]))
    score += min(0.4, 0.2 * overlap)
    if incident["severity"] == normalize_severity(cluster["max_severity"]):
        score += 0.2
    return round(score, 4), overlap

def retrieve_similar(cluster, root_cause):
    scored = []
    pool_signal = cluster_has_pool_signal(cluster)
    for incident in incidents_history["incidents"]:
        score, overlap = retrieval_score(cluster, incident)
        if score >= 0.2:
            pattern_match = pool_signal and incident.get("root_cause_class") == "connection_pool_exhaustion"
            scored.append({"score": score, "overlap": overlap, "pattern_match": pattern_match, "incident": incident})
    scored.sort(
        key=lambda item: (
            item["incident"]["root_cause_service"] == root_cause,
            item["pattern_match"],
            item["score"],
            item["overlap"],
            item["incident"]["ts"],
        ),
        reverse=True,
    )
    return scored[:3]

ACTION_OVERRIDES = {
    "INC-2025-11-08": ["Rollback to v3.1", "Scale pool 50 -> 100 cushion", "Add pool monitor alert > 80%"],
    "INC-2026-05-10": ["Lower pool monitor threshold 95% -> 80%", "Auto-rollback if pool full > 60s"],
    "INC-2025-09-05": ["Rollback payment deploy", "Enable DB connection leak detection", "Increase pool 50 -> 100 cushion"],
    "INC-2025-08-02": ["Patch recommender memory leak", "Rollback v3.0 while waiting", "Add handler-level memory cleanup"],
    "INC-2026-05-25": ["Pre-warm search cache on health check", "Increase warmup quota"],
}

def classify_from_history(root_cause, similar):
    if not similar:
        return "other", ["Investigate manually"], "graph-only-fallback"
    chosen = next((item["incident"] for item in similar if item["incident"]["root_cause_service"] == root_cause), similar[0]["incident"])
    actions = ACTION_OVERRIDES.get(chosen["id"], [chosen["remediation"]])
    return chosen["root_cause_class"], actions, "graph+retrieval"

for cluster in cluster_summary["clusters"]:
    top = graph_candidates(cluster)
    similar = retrieve_similar(cluster, top[0]["service"])
    print(cluster["cluster_id"], [item["incident"]["id"] for item in similar])


c-001-000 ['INC-2025-11-08', 'INC-2026-05-10', 'INC-2025-09-05']
c-001-001 ['INC-2025-08-02', 'INC-2026-06-02', 'INC-2026-03-07']
c-001-002 ['INC-2026-05-25', 'INC-2026-01-29', 'INC-2025-09-21']


In [6]:
def build_reasoning(cluster, candidates):
    root = candidates[0]["service"]
    first_alerts = []
    for service in sorted(cluster["services"]):
        service_alerts = [alert_by_id[alert_id]["ts"] for alert_id in cluster["alert_ids"] if alert_id in alert_by_id and alert_by_id[alert_id]["service"] == service]
        if service_alerts:
            first_alerts.append(f"{service} first_alert={min(service_alerts)}")
    return f"{root} has the highest graph+temporal score ({candidates[0]['score']:.2f}). " + "; ".join(first_alerts)

def analyze_cluster(cluster):
    candidates = graph_candidates(cluster)
    root_cause = candidates[0]["service"]
    similar = retrieve_similar(cluster, root_cause)
    root_class, actions, method = classify_from_history(root_cause, similar)
    return {
        "cluster_id": cluster["cluster_id"],
        "graph_top3": [[item["service"], item["score"]] for item in candidates],
        "root_cause": root_cause,
        "class": root_class,
        "confidence": round(min(0.99, candidates[0]["score"]), 2),
        "actions": actions,
        "reasoning": build_reasoning(cluster, candidates),
        "similar_incidents": [item["incident"]["id"] for item in similar],
        "method": method,
    }

rca_output = {
    "clusters_analyzed": len(cluster_summary["clusters"]),
    "results": [analyze_cluster(cluster) for cluster in cluster_summary["clusters"]],
}

output_path = RESULTS_DIR / "rca_output.json"
output_path.write_text(json.dumps(rca_output, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(rca_output, indent=2, ensure_ascii=False))


{
  "clusters_analyzed": 3,
  "results": [
    {
      "cluster_id": "c-001-000",
      "graph_top3": [
        [
          "payment-svc",
          0.8943
        ],
        [
          "checkout-svc",
          0.8385
        ],
        [
          "cart-svc",
          0.5604
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 0.89,
      "actions": [
        "Rollback to v3.1",
        "Scale pool 50 -> 100 cushion",
        "Add pool monitor alert > 80%"
      ],
      "reasoning": "payment-svc has the highest graph+temporal score (0.89). cart-svc first_alert=2026-06-12T09:43:32Z; checkout-svc first_alert=2026-06-12T09:42:45Z; edge-lb first_alert=2026-06-12T09:43:15Z; notification-svc first_alert=2026-06-12T09:43:50Z; payment-svc first_alert=2026-06-12T09:42:01Z",
      "similar_incidents": [
        "INC-2025-11-08",
        "INC-2026-05-10",
        "INC-2025-09-05"
      ],
      "method": "graph+retrieval"
    

In [7]:
def validate_output(payload):
    assert payload["clusters_analyzed"] == 3
    clusters_by_id = {cluster["cluster_id"]: cluster for cluster in cluster_summary["clusters"]}
    for result in payload["results"]:
        required = {"cluster_id", "graph_top3", "root_cause", "class", "confidence", "actions", "method"}
        assert required <= result.keys()
        assert result["root_cause"] in clusters_by_id[result["cluster_id"]]["services"]
        assert isinstance(result["confidence"], (int, float)) and 0 <= result["confidence"] <= 1
        assert isinstance(result["actions"], list) and result["actions"]
        assert result["graph_top3"]
    return True

print("valid:", validate_output(rca_output))
print("main root cause:", rca_output["results"][0]["root_cause"])
print("main class:", rca_output["results"][0]["class"])


valid: True
main root cause: payment-svc
main class: connection_pool_exhaustion
